# Unitree B1 Leg Forward Kinematics

Closed-form forward kinematics matching `b1_genesis.urdf`. Joint order is `FR, FL, RR, RL`, with each leg ordered as hip, thigh, calf. Angles are radians and positions are metres in the URDF `trunk` frame.

For each leg, let $q_h,q_t,q_c$ be its joint angles and define

$$r=0.35\cos q_t+0.35\cos(q_t+q_c).$$

Then

$$\begin{aligned}x &= x_0-0.35\sin q_t-0.35\sin(q_t+q_c),\\y &= y_0+s(0.12675)\cos q_h+r\sin q_h,\\z &= s(0.12675)\sin q_h-r\cos q_h,\end{aligned}$$

where $s=+1$ for left legs and $s=-1$ for right legs. The foot-link orientation is $R_x(q_h)R_y(q_t+q_c)$.

In [5]:
from typing import Dict
import numpy as np

LEG_ORDER = ("FR", "FL", "RR", "RL")
HIP_ORIGINS = {
    "FR": ( 0.3455, -0.0720, 0.0),
    "FL": ( 0.3455,  0.0720, 0.0),
    "RR": (-0.3455, -0.0720, 0.0),
    "RL": (-0.3455,  0.0720, 0.0),
}
SIDE_SIGN = {"FR": -1.0, "FL": 1.0, "RR": -1.0, "RL": 1.0}
HIP_LINK, THIGH_LINK, CALF_LINK = 0.12675, 0.35, 0.35

In [6]:
def _rotation_and_quaternion(q_hip: np.ndarray, q_pitch: np.ndarray):
    """Return R_x(q_hip) R_y(q_pitch) and quaternion [x,y,z,w]."""
    ch, sh = np.cos(q_hip), np.sin(q_hip)
    cp, sp = np.cos(q_pitch), np.sin(q_pitch)
    shape = q_hip.shape

    rotation = np.empty(shape + (3, 3), dtype=np.result_type(q_hip, float))
    rotation[..., 0, :] = np.stack((cp, np.zeros_like(cp), sp), axis=-1)
    rotation[..., 1, :] = np.stack((sh * sp, ch, -sh * cp), axis=-1)
    rotation[..., 2, :] = np.stack((-ch * sp, sh, ch * cp), axis=-1)

    hh, hp = 0.5 * q_hip, 0.5 * q_pitch
    quaternion = np.stack((
        np.sin(hh) * np.cos(hp),
        np.cos(hh) * np.sin(hp),
        np.sin(hh) * np.sin(hp),
        np.cos(hh) * np.cos(hp),
    ), axis=-1)
    return rotation, quaternion


def b1_foot_forward_kinematics(joint_positions) -> Dict[str, Dict[str, np.ndarray]]:
    """Return all foot poses for input shaped (12,) or (...,12)."""
    q = np.asarray(joint_positions, dtype=float)
    if q.ndim == 0 or q.shape[-1] != 12:
        raise ValueError(f"Expected shape (...,12); got {q.shape}")

    result = {}
    for i, leg in enumerate(LEG_ORDER):
        q_hip, q_thigh, q_calf = np.moveaxis(q[..., 3*i:3*i+3], -1, 0)
        s = SIDE_SIGN[leg]
        x0, y0, z0 = HIP_ORIGINS[leg]
        ch, sh = np.cos(q_hip), np.sin(q_hip)
        pitch = q_thigh + q_calf
        radial = THIGH_LINK*np.cos(q_thigh) + CALF_LINK*np.cos(pitch)

        position = np.stack((
            x0 - THIGH_LINK*np.sin(q_thigh) - CALF_LINK*np.sin(pitch),
            y0 + s*HIP_LINK*ch + radial*sh,
            z0 + s*HIP_LINK*sh - radial*ch,
        ), axis=-1)
        rotation, quaternion = _rotation_and_quaternion(q_hip, pitch)
        result[leg] = {
            "position": position,
            "rotation": rotation,
            "quaternion_xyzw": quaternion,
        }
    return result

## Default standing pose

In [7]:
q_stand = np.array([
    -0.15, 0.67, -1.32,  # FR
     0.15, 0.67, -1.32,  # FL
    -0.15, 0.9, -1.32,  # RR
     0.15, 0.9, -1.32,  # RL
])
poses = b1_foot_forward_kinematics(q_stand)
for leg, pose in poses.items():
    print(f"{leg}: position = {np.round(pose['position'], 6)}")

FR: position = [ 0.33997  -0.279961 -0.527816]
FL: position = [ 0.33997   0.279961 -0.527816]
RR: position = [-0.476948 -0.277597 -0.512172]
RL: position = [-0.476948  0.277597 -0.512172]


## Batched input

Leading dimensions are preserved, which permits vectorized evaluation of many environments.

In [8]:
q_batch = np.stack((q_stand, q_stand + 0.01))  # shape: (2,12)
batch_poses = b1_foot_forward_kinematics(q_batch)
print("FR batch position shape:", batch_poses["FR"]["position"].shape)
print(batch_poses["FR"]["position"])

FR batch position shape: (2, 3)
[[ 0.33997015 -0.27996108 -0.52781641]
 [ 0.33162311 -0.27495074 -0.53184326]]
